<a href="https://colab.research.google.com/github/kerlasaitejaswi17-debug/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search


In [3]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [6]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

Aug 8, 2025 ... TCS BPS Hiring - Batch of 2026. TCS BPS is bringing inspiring entry-level opportunities for Arts and Commerce graduates. As part of Business Processing ... Feb 17, 2026 ... TCS NQT Hiring Announced 2026,2025,2024. 11K views · 3 months ago ...more. Ashish Kumar. 123K. Subscribe. 363. Share. Save. Report. Comments. Mar 11, 2026 ... Tata Consultancy Services (TCS) has officially annou


In [5]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.1 MB/s eta 0:00:00


In [7]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_987/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [8]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '0a1e24ee-60b7-4f51-88b9-0aa662cb7772', 'type': 'tool_call'}]

[2] ToolMessage
    Content: This walk-in hiring drive in Chennai is focused on candidates with strong SAP BTP implementation, SAP Cloud Platform, integration services, extension development, and SAP ecosystem experience. TCS has announced a job notification for the post of HR Operations. A student from various disciplines can 

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I couldn\'t find a specific "2026 hiring quota" for TCS in my search results. The information available points to specific hiring drives and opportunities for various roles and experience levels, including a "TCS Walk in drive 2026" for students, but not an overall quota.'


In [9]:
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I am sorry, I cannot directly access the content of a URL. My `web_search` tool is designed to search the web for information based on a query, not to browse specific URLs. Additionally, the domain `this-domain-does-not-exist-12345.example.com` appears to be a placeholder 


In [10]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [11]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))

aws, spring boot


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Expected: low score (~3-4/10), rationale about lack of specificity / cultural fit


**Score: 2/10**

**Rationale:** Extremely generic and self-serving; shows no research into 'Digital' or genuine interest beyond basic compensation and size, failing to address the "Digital" aspect of the question.


In [12]:
!pip install -U langchain-google-genai

In [14]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_987/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [16]:
import json, pathlib
profiles = json.loads(pathlib.Path('./sample_data/student_profiles.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {p['target_company']}. "
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')



Student 1: Ravi Kumar — CSE CGPA 8.2 → TCS Digital

  [0] HumanMessage
      I am Ravi Kumar, B.Tech CSE CGPA 8.2, skills: Python, Java, SQL, Git. Target: TCS Digital. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': "Hello Ravi! It's great to hear about your profile.\n\nHere are 3 mock interview questions for a TCS Digital role:\n\n1.  Explain the difference between a `LinkedList` and an `ArrayList` in Java. When would you prefer one over the other?\n2.  What are the four pillars of Ob

Student 2: Sneha Reddy — ECE CGPA 7.6 → Cognizant

  [0] HumanMessage
      I am Sneha Reddy, B.Tech ECE CGPA 7.6, skills: C++, Python, MATLAB, Verilog. Target: Cognizant. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Sneha, nice to mee

In [17]:
# Pass a bad URL — see how agent recovers
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': "I wasn't able to fetch the JD from the URL you provided. Please check the link and try again.", 'extras': {'signature': 'Cq8EAQw51seSRi5Mf9xk3Xd49DCK+hY8Ycpc/MGF7zPkXMgGAO8J/FWFOEcXmD16N1954DC9zhK5dUtpvJ5bnAMxYTvWck/85y4a6uXxpaufBwXl9TaEzD0gVhkRurFEtb5cyGEOIozmlNgvf7pouw+H
